# E - Explore: Eksplorasi Data Analitik IDM 2024
## Kelompok 3 - II4013 Data Analytics

Notebook ini murni melakukan analisis eksploratif data (EDA) pada dataset bersih `idm_2024_modeling.csv` hasil dari tahap Scrub.

### Alur Eksplorasi:
1. **Descriptive Statistics**: Analisis ringkasan statistik variabel numerik utama.
2. **Analisis Wilayah Tertinggal (RQ1)**: Mengidentifikasi provinsi dan kabupaten dengan tingkat ketertinggalan paling tinggi.
3. **Proporsi & Segmentasi Status Desa (RQ2)**: Menganalisis sebaran status desa secara nasional.
4. **Analisis Dimensi Penghambat (RQ3)**: Menganalisis gap dimensi indeks untuk menentukan dimensi penentu ketertinggalan.
5. **Identifikasi Potensi Desa (RQ4)**: Pemetaan desa-desa yang berpotensi besar naik kelas ke tingkat berikutnya.
6. **Visualisasi Korelasi & Hubungan**: Heatmap korelasi dan visualisasi hubungan antar-indeks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi tampilan grafik
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (10, 5),
    'axes.titlesize': 13,
    'axes.labelsize': 11
})
sns.set_theme(style='whitegrid', palette='muted')

print('[OK] Library dan konfigurasi grafik siap.')

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())
PROCESSED_DATA_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'idm_2024_modeling.csv')

df = pd.read_csv(PROCESSED_DATA_PATH, dtype={'KODE_DESA': str, 'KODE_PROV': str, 'KODE_KAB': str, 'KODE_KEC': str})
print(f'Memuat dataset modeling: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head(3)

In [ ]:
print('--- Statistik Deskriptif Variabel Utama ---')
fitur_numerik = ['IKS_2024', 'IKE_2024', 'IKL_2024', 'NILAI_IDM_2024', 'jumlah_rekomendasi', 'gap_iks_ike', 'gap_iks_ikl']
display(df[fitur_numerik].describe())

print('\n--- Distribusi Kolom Kategorikal ---')
print(df['STATUS_IDM_2024'].value_counts())
print('\nModus status desa:', df['STATUS_IDM_2024'].mode()[0])

### 1. Wilayah dengan Ketertinggalan Paling Tinggi (RQ1 & Obj1)
Kita mengukur ketertinggalan suatu provinsi dengan menghitung persentase desa berstatus **Tertinggal** dan **Sangat Tertinggal** terhadap total desa di provinsi tersebut.

In [ ]:
# Menghitung persentase desa tertinggal/sangat tertinggal per provinsi
df['is_tertinggal'] = df['STATUS_IDM_2024'].isin(['TERTINGGAL', 'SANGAT TERTINGGAL'])

prov_tertinggal = df.groupby('NAMA_PROVINSI').agg(
    total_desa=('KODE_DESA', 'count'),
    desa_tertinggal=('is_tertinggal', 'sum')
).reset_index()
prov_tertinggal['persentase_tertinggal'] = (prov_tertinggal['desa_tertinggal'] / prov_tertinggal['total_desa'] * 100).round(2)
prov_tertinggal = prov_tertinggal.sort_values('persentase_tertinggal', ascending=False)

print('--- Top 10 Provinsi dengan Persentase Ketertinggalan Tertinggi ---')
display(prov_tertinggal.head(10))

# Plotting
plt.figure(figsize=(12, 6))
sns.barplot(data=prov_tertinggal.head(10), x='persentase_tertinggal', y='NAMA_PROVINSI', palette='Reds_r')
plt.title('Top 10 Provinsi dengan Rasio Desa Tertinggal/Sangat Tertinggal Tertinggi')
plt.xlabel('Rasio Desa Tertinggal (%)')
plt.ylabel('Nama Provinsi')
plt.tight_layout()
plt.show()

### 2. Proporsi dan Segmentasi Desa secara Nasional (RQ2 & Obj2)
Kita menganalisis proporsi desa berdasarkan lima kategori status IDM 2024 secara nasional.

In [ ]:
status_counts = df['STATUS_IDM_2024'].value_counts().reset_index()
status_counts.columns = ['STATUS_IDM_2024', 'JUMLAH']
status_counts['PERSENTASE'] = (status_counts['JUMLAH'] / len(df) * 100).round(2)

print('--- Sebaran Proporsi Status Desa Nasional 2024 ---')
display(status_counts)

# Plotting
plt.figure(figsize=(8, 5))
order = ['SANGAT TERTINGGAL', 'TERTINGGAL', 'BERKEMBANG', 'MAJU', 'MANDIRI']
sns.barplot(data=status_counts, x='STATUS_IDM_2024', y='JUMLAH', order=order, palette='viridis')
plt.title('Proporsi Status Indeks Desa Membangun (IDM) Nasional 2024')
plt.xlabel('Status Desa')
plt.ylabel('Jumlah Desa')
plt.xticks(rotation=15)
for idx, row in status_counts.iterrows():
    # Get index in the ordered list
    if row['STATUS_IDM_2024'] in order:
        pos = order.index(row['STATUS_IDM_2024'])
        plt.text(pos, row['JUMLAH'] + 500, f"{row['PERSENTASE']}%", ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### 3. Dimensi Indeks IDM Penghambat Terbesar (RQ3 & Obj3)
Kita menganalisis dimensi terendah pada tingkat nasional dan desa-desa tertinggal untuk mengetahui pilar pembangunan mana yang paling sering menghambat kemajuan desa.

In [ ]:
print('--- Distribusi Dimensi Terendah Nasional ---')
dimensi_terendah_counts = df['dimensi_terendah'].value_counts().reset_index()
dimensi_terendah_counts.columns = ['DIMENSI', 'JUMLAH']
dimensi_terendah_counts['PERSENTASE'] = (dimensi_terendah_counts['JUMLAH'] / len(df) * 100).round(2)
display(dimensi_terendah_counts)

print('\n--- Distribusi Dimensi Terendah pada Desa Tertinggal & Sangat Tertinggal ---')
df_tertinggal_only = df[df['STATUS_IDM_2024'].isin(['TERTINGGAL', 'SANGAT TERTINGGAL'])]
dim_tertinggal_counts = df_tertinggal_only['dimensi_terendah'].value_counts().reset_index()
dim_tertinggal_counts.columns = ['DIMENSI', 'JUMLAH']
dim_tertinggal_counts['PERSENTASE'] = (dim_tertinggal_counts['JUMLAH'] / len(df_tertinggal_only) * 100).round(2)
display(dim_tertinggal_counts)

# Plotting perbandingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=dimensi_terendah_counts, x='DIMENSI', y='PERSENTASE', ax=axes[0], palette='Blues_r')
axes[0].set_title('Dimensi Terendah (Faktor Penghambat) Nasional')
axes[0].set_ylabel('Persentase Desa (%)')
axes[0].set_xlabel('Dimensi')

sns.barplot(data=dim_tertinggal_counts, x='DIMENSI', y='PERSENTASE', ax=axes[1], palette='Oranges_r')
axes[1].set_title('Dimensi Terendah pada Desa Tertinggal & Sangat Tertinggal')
axes[1].set_ylabel('Persentase Desa (%)')
axes[1].set_xlabel('Dimensi')

plt.tight_layout()
plt.show()

### 4. Desa dengan Potensi Berkembang Terbesar (RQ4 & Obj4)
Desa dengan potensi berkembang tertinggi adalah desa yang skor IDM-nya berada sangat dekat di bawah batas kelas berikutnya, serta memiliki kesenjangan ekonomi/sosial yang relatif kecil sehingga sedikit intervensi terarah dapat membuat desa tersebut naik kelas.

In [ ]:
# Menentukan batas skor IDM (Sangat Tertinggal < 0.4907 <= Tertinggal < 0.5989 <= Berkembang < 0.7072 <= Maju < 0.8155 <= Mandiri)
# Desa berpotensi berkembang ke kelas berikutnya adalah yang nilainya berada di batas 0.02 di bawah ambang batas status di atasnya.
def check_potensi(row):
    nilai = row['NILAI_IDM_2024']
    status = row['STATUS_IDM_2024']
    if status == 'SANGAT TERTINGGAL' and 0.4707 <= nilai < 0.4907:
        return 'Potensi ke TERTINGGAL'
    elif status == 'TERTINGGAL' and 0.5789 <= nilai < 0.5989:
        return 'Potensi ke BERKEMBANG'
    elif status == 'BERKEMBANG' and 0.6872 <= nilai < 0.7072:
        return 'Potensi ke MAJU'
    elif status == 'MAJU' and 0.7955 <= nilai < 0.8155:
        return 'Potensi ke MANDIRI'
    return 'Biasa'

df['potensi_naik'] = df.apply(check_potensi, axis=1)
potensi_counts = df['potensi_naik'].value_counts().reset_index()
potensi_counts.columns = ['KATEGORI_POTENSI', 'JUMLAH']
display(potensi_counts)

# Contoh desa berkembang dengan potensi besar ke kelas berikutnya
print('\n--- Contoh Desa Berkembang dengan Potensi Naik Kelas Paling Dekat (ke MAJU) ---')
df_potensi_maju = df[df['potensi_naik'] == 'Potensi ke MAJU'].sort_values('NILAI_IDM_2024', ascending=False)
display(df_potensi_maju[['NAMA_PROVINSI', 'NAMA_KABUPATEN', 'NAMA_KECAMATAN', 'NAMA_DESA', 'NILAI_IDM_2024', 'dimensi_terendah', 'jumlah_rekomendasi']].head(5))

### 5. Hubungan Antar-Indeks Komposit dan Korelasi Rekomendasi (RQ5 & Obj5)
Kita menganalisis korelasi antar-fitur indeks numerik untuk memahami hubungan ketahanan desa dan bagaimana rekomendasi berkorelasi dengan kondisi desa.

In [ ]:
# Korelasi heatmap
plt.figure(figsize=(8, 6))
corr_matrix = df[['IKS_2024', 'IKE_2024', 'IKL_2024', 'NILAI_IDM_2024', 'jumlah_rekomendasi', 'gap_iks_ike', 'gap_iks_ikl']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Heatmap Korelasi Indikator IDM & Rekomendasi')
plt.tight_layout()
plt.show()

# Scatter plot hubungan IKE dan IKS berdasarkan Status Desa
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(5000, random_state=42), x='IKE_2024', y='IKS_2024', hue='STATUS_IDM_2024', alpha=0.5, palette='Set1')
plt.title('Hubungan Indeks Ketahanan Ekonomi (IKE) vs Indeks Ketahanan Sosial (IKS) (Sampel 5.000 Desa)')
plt.xlabel('IKE 2024')
plt.ylabel('IKS 2024')
plt.legend(title='Status IDM')
plt.tight_layout()
plt.show()

### 6. Ringkasan Statistik Indeks dan Modus Status per Provinsi
Bagian ini mengekstrak profil pembangunan tingkat daerah untuk melihat nilai median indikator dan modus status desa di setiap provinsi.

In [ ]:
# Median numerik dan Modus status per provinsi
def get_mode(series):
    return series.mode()[0] if not series.dropna().empty else np.nan

prov_stats = df.groupby('NAMA_PROVINSI').agg(
    IKS_median=('IKS_2024', 'median'),
    IKE_median=('IKE_2024', 'median'),
    IKL_median=('IKL_2024', 'median'),
    NILAI_IDM_median=('NILAI_IDM_2024', 'median'),
    STATUS_IDM_modus=('STATUS_IDM_2024', get_mode),
    jumlah_desa=('KODE_DESA', 'count')
).reset_index()

print('--- Ringkasan Statistik Provinsi (Diurutkan berdasarkan Median Nilai IDM Terendah) ---')
display(prov_stats.sort_values('NILAI_IDM_median', ascending=True))

# Simpan statistik wilayah untuk keperluan visualisasi dashboard
prov_stats_path = os.path.join(BASE_DIR, 'data', 'processed', 'provinsi_stats_summary.csv')
prov_stats.to_csv(prov_stats_path, index=False)
print(f'\n[SUCCESS] Ringkasan statistik provinsi disimpan ke: {prov_stats_path}')